In [4]:
import requests
import pandas as pd
import ast

from keys import TMDB_API_KEY

BASE_URL = "https://api.themoviedb.org/3"

# Load genre lookup without hardcoding
df_genres = pd.read_csv("genres.csv") # Listed id, genre name
genre_lookup = dict(zip(df_genres["id"], df_genres["name"])) # Combine ID and word into a list


In [11]:
def search_movie(term, year=None):
    url = f"{BASE_URL}/search/movie"    

    params = {
        "api_key": TMDB_API_KEY,
        "query": term
    }

    # Include year in search only if the user includes it
    if year:
       params["primary_release_year"] = year

    response = requests.get(url, params=params)

    # Check to make sure results are actually being sent
    if response.status_code != 200:
        print("Error: ", response.status_code)
        print(response.text)
        return []


    data = response.json()

    # If something is returned that isn't expected
    if "results" not in data or data["results"] is None:
        print("TMDB returned no results.")
        print(data)
        return []
    
    results = data.get("results", [])

    movies = []
    for m in results:
        movies.append({
            "id": m["id"],
            "title": m["title"],
            "year": (m.get("release_date") or "")[:4],
            "overview": m.get("overview", ""),
            "genre_ids": m.get("genre_ids", [])
        })



    return movies

In [14]:
# Testing Movie lookup without hardcoding
term = input("Search for a movie: ")
year = input("Enter a year to search by (optional): ")

# Check to see if year was entered
if year == "":
    year = None
movies = search_movie(term, year)

print('\nTop Results\n')

for m in movies[:3]:
    print(f"Title: {m['title']}")
    print(f"Year: {m['year']}")
    print(f"Genres (IDs): {m['genre_ids']}")
    print(f"Overview: {m['overview'][:150]}...")
    print("-" * 50)


Top Results

Title: Dune
Year: 2021
Genres (IDs): [878, 12, 10752]
Overview: Paul Atreides, a brilliant and gifted young man born into a great destiny beyond his understanding, must travel to the most dangerous planet in the un...
--------------------------------------------------
Title: Planet Dune
Year: 2021
Genres (IDs): [878, 27, 12]
Overview: A crew on a mission to rescue a marooned base on a desert planet turns deadly when the crew finds themselves hunted and attacked by the planet’s apex ...
--------------------------------------------------
Title: Devil In Dune
Year: 2021
Genres (IDs): [878, 28, 27]
Overview: The film tells the story of a near future in which the earth is irreversibly sandy, as human emissions and rubbish exceed its carrying capacity. As pl...
--------------------------------------------------


In [ ]:
def get_director(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}/credits"
    params = {"api_key": TMDB_API_KEY}

    response = requests.get(url, params=params)

    # Check to make sure API is returning needed information
    if response.status_code != 200:
        print("Error getting director info:", response.status_code)
        return "Unknown"
    
    data = response.json()

    # Get crew data
    crew = data.get("crew", [])

    # Loop through crew list to find director
    for person in crew:
        if person.get("job") == "Director":
            return person.get("name", "Unknown")
        
    return "Unknown"

In [29]:
def submit_review(movie_id, rating, movie_title, genre_ids=None):
    # Validate User review
    if rating < 1 or rating > 5:
        print("You must pick between 1 and 5 stars.")
        return
    
    # Load existing reviews (if any)
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["movie_id", "rating", "genre_ids"])

    # Check to see if there is genre ids, add space if none
    if genre_ids is None:
        genre_ids_str = ""
    else:
        genre_ids_str = str(genre_ids)

    # Add a new row for the new review
    new_row = {
        "movie_id": movie_id,
        "title": movie_title,
        "rating": float(rating),
        "genre_ids": genre_ids_str
    }
    # Add new review to the bottom of reviews.csv
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Save new review to reviews.csv
    df.to_csv("reviews.csv", index=False)
    print("Your review was saved!")

In [5]:
# Test submit_review without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)

# Show search results
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# User selects one from list of results
choice = int(input("Select a movie from the list using the number: "))
selected = movies[choice]

# User enters rating
rating = int(input("Enter your star review (1-5): "))

# Submit review
submit_review(selected["id"], rating, selected["genre_ids"])


1. The Toxic Avenger Unrated (2025)
2. The Toxic Avenger (1984)
3. The Toxic Avenger: The Musical (2018)
4. Citizen Toxie: The Toxic Avenger IV (2001)
5. The Toxic Avenger Part II (1989)
6. The Toxic Avenger Part III: The Last Temptation of Toxie (1989)
7. Heart of Fartness: Troma's First VR Experience Starring the Toxic Avenger (2017)
8. Dr. Platypus and the Toxic Avengers: The Probe ()
Your review was saved!


In [17]:
def get_movie_details(movie_id):

    # Get movie details from TMDB
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        "api_key": TMDB_API_KEY
    }
    response = requests.get(url, params=params)

    # Check to make sure request is fufilled by TMDB API
    if response.status_code != 200:
        print("TMDB request error:", response.status_code)
        return {}
    
    # Return details in JSON format
    return response.json()

In [7]:
# Testing get_movie_details
term = input("Search for a movie: ")
movies = search_movie(term)

# Show results of search
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# Have user pick one to view details
selected = movies[int(input("Select a movie by the number on the list: ")) - 1]
details = get_movie_details(selected["id"])

print(details)

1. The Matrix (1999)
2. The Matrix Reloaded (2003)
3. The Matrix Revolutions (2003)
4. The Matrix Resurrections (2021)
5. The Matrix Revisited (2001)
6. The Matrix Recalibrated (2004)
7. Making "The Matrix" (1999)
8. The Matrix: Generation (2023)
9. The Matrix Reloaded Revisited (2004)
10. The Matrix 5 ()
11. The Matrix (2004)
12. The Religion Matrix (2021)
13. Sex and the Matrix (2000)
14. The Matrix Revolutions Revisited (2004)
15. Making 'Enter the Matrix' (2003)
16. The Matrix ()
17. Exit the matrix (2026)
18. The Matrix Revolutions: Super Big Mini Models (2004)
19. The Living Matrix (2009)
20. The Matrix REdeZIONized (2022)
{'adult': False, 'backdrop_path': '/tlm8UkiQsitc8rSuIAscQDCnP8d.jpg', 'belongs_to_collection': {'id': 2344, 'name': 'The Matrix Collection', 'poster_path': '/bV9qTVHTVf0gkW0j7p7M0ILD4pG.jpg', 'backdrop_path': '/bRm2DEgUiYciDw3myHuYFInD7la.jpg'}, 'budget': 63000000, 'genres': [{'id': 28, 'name': 'Action'}, {'id': 878, 'name': 'Science Fiction'}], 'homepage': 'ht

In [8]:
def movies_with_genre(df_movies, genre_id):
    # Filter results by user selected genre
    return df_movies[df_movies["genre_ids"].apply(lambda g: genre_id in g)]

In [9]:
# Test movies_with_genre() filtering
term = input("Search for a movie: ")
movies = search_movie(term)
df_movies = pd.DataFrame(movies)
print(df_movies)

# Prompt user for genre for filtering
genre_name = input("Enter a genre to filter by: ")

# Load genre lookup
df_genres = pd.read_csv("genres.csv")
df_genres.columns = df_genres.columns.str.strip()

# Convert names back to ID
genre_id = df_genres[df_genres["name"] == genre_name]["id"].iloc[0]

# Filter movies by selected genre
filtered = movies_with_genre(df_movies, genre_id)
print(filtered)

        id                      title  year  \
0     2787                Pitch Black  2000   
1   880234                Pitch Black  2021   
2   707403        Pitch Black Panacea  2020   
3  1057915                Pitch Black  2023   
4   116065          Pitch Black Heist  2012   
5   244839           Into Pitch Black  2000   
6   254631     Pitch Black: Slam City  2000   
7  1257220                Pitch-Black  1973   
8   405047                     Siyaah  2013   
9     2789  The Chronicles of Riddick  2004   

                                            overview             genre_ids  
0  When their ship crash-lands on a remote planet...         [53, 878, 28]  
1  Weary from his wife's repeated suicide attempt...            [53, 9648]  
2  Amy and Carl both have lazy eyes. Amy’s left a...              [16, 35]  
3  Pitch Black takes us inside the claustrophobic...              [99, 18]  
4  Liam and Michael are professional safe cracker...              [80, 18]  
5  After the events 

In [ ]:
def generate_movieboard(top_n=10):
    # Read reviews.csv and check to see if there is reviews
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        print("No reviews yet.")
        return
    
    # No reviews present?
    if df.empty:
        print("No reviews yet.")
        return
    
    # Convert genre_ids back to lists
    df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if x else []) # Add space if no genre id is present
    
    # group movies by ID
    grouped_ID = df.groupby("movie_id")["rating"].agg(["mean", "count"]).reset_index()
    grouped_ID = grouped_ID.sort_values(by=["mean", "count"], ascending=[False, False])

    movieboard = []

    # Show top movies based on set value (n)
    print("Top movies:")
    for i, row in grouped_ID.head(top_n).iterrows():
        movie_id = int(row["movie_id"])
        avg = row["mean"]
        review_count = int(row["count"])

        # Get movie title from TMDB 
        details = get_movie_details(movie_id)
        title = details.get("title", "Unknown title")

        # Convert TMDB Genre IDs to words
        tmdb_genres = details.get("genres", [])
        genre_names = [genre_lookup[g["id"]] for g in tmdb_genres if g["id"] in genre_lookup]

        # Add placeholder if no genre tags present
        if not genre_names:
            genre_names = ["No genre tags availible"]

        
        # Get director name using get_director()
        director = get_director(movie_id)

        # Assemble movieboard entry
        movie_entry = {
            "id": movie_id,
            "title": title,
            "avg_rating": float(round(avg, 2)),
            "review_count": review_count,
            "genres": genre_names,
            "director": director

        }

        # Add entry to final board
        movieboard.append(movie_entry)

        return movieboard
    

In [28]:
# Test generate_movieboard
generate_movieboard()

Top movies:


[{'id': 7512,
  'title': 'Idiocracy',
  'avg_rating': 5.0,
  'review_count': 1,
  'genres': ['Comedy', 'Science Fiction', 'Adventure'],
  'director': 'Mike Judge'}]

In [23]:
# Test csv loading
df = pd.read_csv("reviews.csv")
print(df)

   movie_id  rating      genre_ids
0      7512     NaN  [35, 878, 12]
1      7512     NaN  [35, 878, 12]
2      7512     5.0  [35, 878, 12]
3    157336     3.0  [12, 18, 878]
